# SDK + Direct API Walkthrough

> Plan note: for free-plan accounts, keep GPU usage at 1 and prefer non-parallel real modes (`lumi_v1_6` or `aws_v1_6`).

> This notebook uses CLI-based auth (`qas auth login`) and then runs one compression flow through the SDK and one direct REST fetch for parity checks.

## Install the SDK in this kernel
Install the latest public SDK release from PyPI:

```bash
pip install -U qas-sdk
```


In [ ]:
import sys

print("Kernel Python:", sys.executable)
!{sys.executable} -m pip install -U qas-sdk

In [ ]:
import os
import subprocess
from textwrap import dedent


def optional_int(name: str) -> object:
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return None
    try:
        return int(raw)
    except ValueError as exc:
        msg = f"Environment variable {name} must be an integer"
        raise RuntimeError(msg) from exc


def cli_token() -> str:
    try:
        result = subprocess.run(
            ["qas", "auth", "token"],
            capture_output=True,
            check=True,
            text=True,
            timeout=30,
        )
    except (subprocess.CalledProcessError, OSError, subprocess.TimeoutExpired) as exc:
        msg = "Unable to retrieve token from CLI. Run `qas auth login` first."
        raise RuntimeError(msg) from exc
    token = result.stdout.strip()
    if not token:
        msg = "Empty token received from CLI."
        raise RuntimeError(msg)
    return token


BASE_URL = os.getenv("QAS_BASE_URL", "https://qas.qmill.com").rstrip("/")
ACCESS_TOKEN = cli_token()

NUM_GPUS = optional_int("QAS_NUM_GPUS")
ITERATION_MINUTES = optional_int("QAS_ITERATION_MINUTES")
GATE_SET = os.getenv("QAS_GATE_SET")
HPC_MODE = os.getenv("QAS_HPC_MODE")

print(
    dedent(
        f"""
        Using QAS environment: {BASE_URL}
        Auth source: qas auth token
        Compression overrides: num_gpus={NUM_GPUS}, iteration_minutes={ITERATION_MINUTES}, gate_set={GATE_SET}, hpc_mode={HPC_MODE}
        """
    ).strip(),
)

## Compression Parameters (Optional Tuning)
- `hpc_mode`: one of the available modes returned by `get_hpc_mode()`; default is `lumi_v1_6` when omitted.
- `num_gpus`: number of GPUs for real backends; default is `1` when omitted; use `None` to keep the backend default in this notebook.
- `iteration_time_minutes`: max runtime per iteration in minutes.
- `gate_set`: choose from the mode's `gate_set_options` returned by `get_hpc_mode()`.

Tip: prefer explicit parameters in code for users; env vars are handy for local testing.

In [ ]:
# Optional: check Terms & Conditions status. Skip this cell if your environment does not require terms acceptance.
import json
import requests

terms_response = requests.get(
    f"{BASE_URL}/api/v1/terms",
    headers={
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Accept": "application/json",
    },
    timeout=30,
)
terms_response.raise_for_status()
TERMS_STATUS = terms_response.json()

print(
    json.dumps(
        {
            "enabled": TERMS_STATUS.get("enabled"),
            "required_version": TERMS_STATUS.get("required_version"),
            "accepted_version": TERMS_STATUS.get("accepted_version"),
            "compliant": TERMS_STATUS.get("compliant"),
            "document_url": TERMS_STATUS.get("document_url"),
        },
        indent=2,
    )
)

ACCEPT_TERMS = False  # Set to True after reviewing the referenced document.
if ACCEPT_TERMS and not TERMS_STATUS.get("compliant", False):
    accept_payload = {"version": TERMS_STATUS.get("required_version") or None}
    accept_response = requests.post(
        f"{BASE_URL}/api/v1/terms/accept",
        headers={
            "Authorization": f"Bearer {ACCESS_TOKEN}",
            "Accept": "application/json",
        },
        json=accept_payload,
        timeout=30,
    )
    accept_response.raise_for_status()
    print("Accepted terms response:", accept_response.text)
    TERMS_STATUS = accept_response.json()
    print("Terms accepted for the required version.")
elif not TERMS_STATUS.get("compliant", False):
    doc_hint = TERMS_STATUS.get("document_url") or "your assigned Terms document"
    print(
        "Terms not yet accepted. Review",
        doc_hint,
        "and re-run with ACCEPT_TERMS = True to record acceptance.",
    )

In [ ]:
import importlib
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
repo_root_str = str(repo_root)
sys.path = [p for p in sys.path if p and not str(p).startswith(repo_root_str)]
sys.modules.pop("qas_sdk", None)
qas_sdk_module = importlib.import_module("qas_sdk")
QASClient = qas_sdk_module.QASClient
CompressionJobOptions = qas_sdk_module.CompressionJobOptions

client = QASClient(base_url=BASE_URL)


mod5_4_circuit = dedent(
    """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[5];
    x q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    cx q[0],q[3];
    rz(-pi/4) q[3];
    cx q[0],q[3];
    rz(pi/4) q[0];
    rz(pi/4) q[3];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[2],q[4];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[2],q[4];
    cx q[2],q[3];
    rz(-pi/4) q[3];
    cx q[2],q[3];
    rz(pi/4) q[2];
    rz(pi/4) q[3];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[3],q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[2],q[4];
    rz(-pi/4) q[4];
    cx q[1],q[4];
    rz(pi/4) q[4];
    cx q[2],q[4];
    rz(-pi/4) q[4];
    cx q[1],q[4];
    cx q[1],q[2];
    rz(-pi/4) q[2];
    cx q[1],q[2];
    rz(pi/4) q[1];
    rz(pi/4) q[2];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[2],q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[1],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    rz(pi/4) q[4];
    cx q[1],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    cx q[0],q[1];
    rz(-pi/4) q[1];
    cx q[0],q[1];
    rz(pi/4) q[0];
    rz(pi/4) q[1];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[1],q[4];
    cx q[0],q[4];
    """
).strip()

num_gpus = NUM_GPUS
iteration_minutes = ITERATION_MINUTES
gate_set = GATE_SET
hpc_mode = HPC_MODE

compression_kwargs = {}
if num_gpus is not None:
    compression_kwargs["num_gpus"] = num_gpus

options = CompressionJobOptions(
    iteration_time_minutes=iteration_minutes,
    gate_set=gate_set,
    hpc_mode=hpc_mode,
)
if options.to_payload():
    compression_kwargs["options"] = options

print("SDK authentication complete. Submitting compression job...")
if compression_kwargs:
    print(f"Using compression overrides: {compression_kwargs}")
job = client.submit_compression(mod5_4_circuit, **compression_kwargs)
job_id = job["job_id"]
print(f"Submitted job: {job_id}")

sdk_result = client.wait_for_job(job_id, poll_interval=5, timeout=600)
print("SDK reported status:", sdk_result.get("status"))
circuit_metrics = sdk_result.get("circuit_metrics") or {}
compressed_metrics = sdk_result.get("compressed_metrics") or {}
print("Original gates:", circuit_metrics.get("gate_count"))
print("Compressed gates:", compressed_metrics.get("gate_count"))
print("Compressed circuit:", sdk_result.get("result"))
print("Transpiled circuit:", sdk_result.get("transpiled_circuit"))

In [ ]:
import json

import requests

TOKEN_FOR_API = ACCESS_TOKEN

job_url = f"{BASE_URL}/api/public/v1/circuit-compression/jobs/{job_id}"
api_response = requests.get(
    job_url,
    headers={
        "Authorization": f"Bearer {TOKEN_FOR_API}",
        "Accept": "application/json",
    },
    timeout=30,
)
api_response.raise_for_status()
payload = api_response.json()
print(json.dumps(payload, indent=2, sort_keys=True))